# 03 - Class steering with Gaussian/PCA denoisers

At high noise, a generator cannot yet know fine sample details, but means and
major directions of variation can still provide useful structure. This
notebook first tests that idea on MNIST pixels, then uses the difference between
a target-class and full-data denoiser as a coarse post-hoc steering signal:

$$\Delta D = D_{target\ class} - D_{full\ data}.$$

There is no hand-written anchor or attraction toward a coordinate. The final
2D experiment is the visible counterpart of the high-noise noise-alignment
stage in NA-RFM.


In [ ]:
import sys
from pathlib import Path

NOTES_DIR = Path.cwd()
if not (NOTES_DIR / "toy_notes.py").exists():
    NOTES_DIR = Path("notebooks/toy_data")
sys.path.insert(0, str(NOTES_DIR.resolve()))

from IPython import get_ipython

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from toy_notes import *

SEED = 2026
set_seed(SEED)
device = choose_device()
print(environment_summary(device))


In [ ]:
model, losses, trained_now = load_or_train_model(device=device)
model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)
checkpoint_digest_before = model_state_digest(model)

set_seed(SEED + 301)
steering_data, steering_labels = sample_labeled_mixture(30000, device=device)
steering_stats = estimate_gaussian_stats(steering_data, steering_labels)
set_seed(SEED + 302)
evaluation_data, evaluation_labels = sample_labeled_mixture(30000, device=device)
evaluation_stats = estimate_gaussian_stats(evaluation_data, evaluation_labels)
TARGET_CLASS = 2
set_seed(SEED + 303)
target_reference = sample_class(1800, TARGET_CLASS, device=device)
print("target class:", CLASS_NAMES[TARGET_CLASS])
print("frozen checkpoint digest:", checkpoint_digest_before[:16])


## 1. Why a Gaussian denoiser makes sense at high noise

Write the base point as $x_{noise}=s\epsilon$ with
$\epsilon\sim\mathcal{N}(0,I)$ and `s=1.8`, then divide the path by `t`:

$$\frac{x_t}{t}=x_{data}+s\frac{1-t}{t}\epsilon.$$

The right side is clean data plus Gaussian noise with effective noise level
`sigma_eff=s(1-t)/t`. For a Gaussian data model, the posterior mean is

$$D(y,\sigma)=\mu+C(C+\sigma^2I)^{-1}(y-\mu).$$

If $C=U\Lambda U^\top$, each principal-component coordinate is multiplied by
$\lambda_i/(\lambda_i+\sigma^2)$. Large-variance directions survive; uncertain
small-variance directions shrink toward the mean. This is why PCA is a useful
way to compute the same rule for images.

The factor `s` matters because our base distribution is not unit variance.
At high noise the denoiser trusts coarse mean/covariance statistics; at low noise
it stays close to the observed point.

The full-data Gaussian is deliberately crude because the complete dataset is a
mixture. That limitation is useful: subtracting the full denoiser from the
target-class denoiser isolates a coarse class correction.

[Li, Dai, and Qu (NeurIPS 2024)](https://arxiv.org/abs/2410.24060) report that
diffusion denoisers become increasingly linear in a generalization regime and
that their best linear approximations are close to empirical Gaussian
denoisers. Their study includes FFHQ and checks on CIFAR-10, AFHQ, and
LSUN-Churches. The MNIST calculation below is our teaching demonstration, not
an experiment from that paper. We do not present it as ImageNet evidence; the
later CIFAR-10 notebook supplies the image-generator experiment in this series.


## 2. Does a covariance model retain useful image structure?

MNIST makes all 784 pixel coordinates visible without needing a large network.
We fit low-rank Gaussian models on the training split only: one to all digits
and one to digit 3. The leading covariance directions are `eigendigits`.
On held-out test images, we add known Gaussian noise and compare the noisy input
with both posterior-mean denoisers.


In [ ]:
MNIST_TARGET = 3
MNIST_RANK = 64
MNIST_SIGMA = 0.45

mnist_train_images, mnist_train_labels = load_mnist_split(train=True, device=device)
mnist_test_images, mnist_test_labels = load_mnist_split(train=False, device=device)

full_fit_indexes = torch.linspace(
    0, mnist_train_images.shape[0] - 1, 12000, device=device
).round().long()
class_fit_pool = torch.where(mnist_train_labels == MNIST_TARGET)[0]
class_fit_indexes = class_fit_pool[: min(6000, class_fit_pool.numel())]

set_seed(SEED + 330)
mnist_full_gaussian = fit_low_rank_gaussian(
    mnist_train_images[full_fit_indexes], rank=MNIST_RANK
)
set_seed(SEED + 331)
mnist_class_gaussian = fit_low_rank_gaussian(
    mnist_train_images[class_fit_indexes], rank=MNIST_RANK
)

print("MNIST training images used for full Gaussian:", full_fit_indexes.numel())
print("MNIST digit-3 training images used:", class_fit_indexes.numel())
print("PCA rank:", MNIST_RANK)


In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(10, 3.8))
axes[0, 0].imshow(mnist_full_gaussian.mean.reshape(28, 28).cpu(), cmap="gray", vmin=0, vmax=1)
axes[0, 0].set_title("all-digit mean")
axes[1, 0].imshow(mnist_class_gaussian.mean.reshape(28, 28).cpu(), cmap="gray", vmin=0, vmax=1)
axes[1, 0].set_title("digit-3 mean")
for column in range(1, 6):
    for row, stats, label in [
        (0, mnist_full_gaussian, "all"),
        (1, mnist_class_gaussian, "digit 3"),
    ]:
        component = stats.components[:, column - 1].reshape(28, 28).cpu()
        scale = float(component.abs().max().clamp_min(1e-8))
        axes[row, column].imshow(component, cmap="RdBu_r", vmin=-scale, vmax=scale)
        axes[row, column].set_title(f"{label} PC {column}")
for ax in axes.flat:
    ax.axis("off")
fig.suptitle("Means and leading covariance directions learned from MNIST training images", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
held_out_pool = torch.where(mnist_test_labels == MNIST_TARGET)[0]
held_out_indexes = held_out_pool[:256]
mnist_clean = mnist_test_images[held_out_indexes]
set_seed(SEED + 332)
mnist_noisy = mnist_clean + MNIST_SIGMA * torch.randn_like(mnist_clean)
mnist_full_denoised = low_rank_gaussian_denoise(
    mnist_noisy, MNIST_SIGMA, mnist_full_gaussian
)
mnist_class_denoised = low_rank_gaussian_denoise(
    mnist_noisy, MNIST_SIGMA, mnist_class_gaussian
)

def image_mse(images):
    return float(torch.mean((images - mnist_clean).square()).cpu())

mnist_mse = {
    "noisy input": image_mse(mnist_noisy),
    "all-digit Gaussian": image_mse(mnist_full_denoised),
    "digit-3 Gaussian": image_mse(mnist_class_denoised),
}
mnist_metrics = pd.DataFrame({
    "method": list(mnist_mse),
    "held-out MSE": list(mnist_mse.values()),
    "PSNR (dB)": [10 * np.log10(1.0 / value) for value in mnist_mse.values()],
})
display(mnist_metrics.round(4))

display_count = 10
fig, axes = plt.subplots(4, display_count, figsize=(13, 5.4))
rows = [
    ("clean test image", mnist_clean),
    (f"noisy, sigma={MNIST_SIGMA}", mnist_noisy),
    ("all-digit Gaussian", mnist_full_denoised),
    ("digit-3 Gaussian", mnist_class_denoised),
]
for row, (label, images) in enumerate(rows):
    for column in range(display_count):
        axes[row, column].imshow(images[column].reshape(28, 28).detach().cpu().clamp(0, 1), cmap="gray", vmin=0, vmax=1)
        axes[row, column].axis("off")
    axes[row, 0].set_ylabel(label, rotation=0, ha="right", va="center", labelpad=34)
fig.suptitle("Gaussian/PCA posterior means on held-out digit-3 images", y=1.01)
plt.tight_layout()
plt.show()

mnist_checks = pd.DataFrame({
    "check": ["all-digit Gaussian beats noisy input", "digit-3 Gaussian beats noisy input"],
    "passed": [
        mnist_mse["all-digit Gaussian"] < mnist_mse["noisy input"],
        mnist_mse["digit-3 Gaussian"] < mnist_mse["noisy input"],
    ],
})
display(mnist_checks)
print("MNIST Gaussian denoising verdict:", "PASS" if mnist_checks["passed"].all() else "CHECK OUTPUT")


## 3. How does denoising become a steering signal?

The image example shows what one Gaussian posterior mean preserves and loses.
For steering, we compare two such estimates at the same noisy state. Their
difference asks for class-level structure that the target model favors more
than the full-data model. The gate below limits that correction to the
high-noise part of the trajectory.


In [ ]:
grid_1d = torch.linspace(-4.5, 4.5, 19, device=device)
gx, gy = torch.meshgrid(grid_1d, grid_1d, indexing="xy")
grid = torch.stack([gx.reshape(-1), gy.reshape(-1)], dim=1)
t_value = 0.18
t = torch.full((grid.shape[0],), t_value, device=device)
delta = noise_alignment_delta(grid, t, TARGET_CLASS, steering_stats)

gate_times = torch.linspace(0, 1, 300, device=device)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
axes[0].plot(gate_times.cpu(), window_gate(gate_times, 0.04, 0.35).cpu(), color="#0072B2", lw=2)
axes[0].set(xlabel="t: noise to data", ylabel="gate g(t)", title="Correction is active only at high noise")
axes[0].grid(alpha=0.2)
axes[1].quiver(
    grid[:, 0].cpu(), grid[:, 1].cpu(), delta[:, 0].cpu(), delta[:, 1].cpu(),
    angles="xy", scale_units="xy", scale=1.7, width=0.004, color=CLASS_COLORS[TARGET_CLASS], alpha=0.8,
)
plot_labeled_points(
    steering_data[:2500], steering_labels[:2500], ax=axes[1],
    title=f"D_target - D_full at t={t_value} (clean-estimate coordinates)", alpha=0.18,
)
axes[1].set_xlim(-4.8, 4.8)
axes[1].set_ylim(-4.8, 4.8)
plt.tight_layout()
plt.show()


## 4. Does the correction improve paired samples?

All methods below use the same initial noise, Heun solver, step count, and target
reference. The only change is the noise-alignment strength. The correction is
active for `t in [0.04, 0.35]`, the high-noise part of our noise-to-data path.


In [ ]:
set_seed(SEED + 30)
x0 = sample_base(1800, device=device)

methods = {
    "baseline": base_velocity(model),
    "noise alignment 0.0": make_noise_aligned_velocity(
        model, steering_stats, target_class=TARGET_CLASS, strength=0.0, start=0.04, end=0.35
    ),
}
for strength in [0.4, 0.8, 1.2, 1.6]:
    methods[f"noise alignment {strength:.1f}"] = make_noise_aligned_velocity(
        model, steering_stats, target_class=TARGET_CLASS, strength=strength, start=0.04, end=0.35
    )

rows = []
paths = {}
for name, velocity in methods.items():
    path, seconds = timed_sample(velocity, x0, steps=64, method="heun")
    metrics = evaluate_samples(
        path[-1].to(device), target_reference, evaluation_stats, target_class=TARGET_CLASS
    )
    strength = np.nan if name == "baseline" else float(name.rsplit(" ", 1)[-1])
    rows.append({"method": name, "strength": strength, **metrics, "seconds": seconds})
    paths[name] = path

noise_alignment_table = pd.DataFrame(rows)
display(noise_alignment_table.round(4))

zero_strength_error = float(torch.max(torch.abs(paths["baseline"][-1] - paths["noise alignment 0.0"][-1])))
print(f"paired endpoint max difference, baseline vs zero strength: {zero_strength_error:.2e}")
assert zero_strength_error < 1e-6
assert model_state_digest(model) == checkpoint_digest_before
print("checkpoint digest unchanged after every rollout: True")


In [ ]:
baseline_endpoints = paths["baseline"][-1].to(device)
aligned_endpoints = paths["noise alignment 1.6"][-1].to(device)
baseline_classes = predict_classes(baseline_endpoints, evaluation_stats)
aligned_classes = predict_classes(aligned_endpoints, evaluation_stats)

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.3))
for ax, endpoints, predicted, title in [
    (axes[0], baseline_endpoints, baseline_classes, "Baseline endpoints"),
    (axes[1], aligned_endpoints, aligned_classes, "Noise alignment, strength 1.6"),
]:
    ax.scatter(target_reference[:, 0].cpu(), target_reference[:, 1].cpu(), s=8, alpha=0.12, color="#555555", label="target examples")
    for class_id, class_name in enumerate(CLASS_NAMES):
        subset = endpoints[predicted == class_id].cpu()
        ax.scatter(subset[:, 0], subset[:, 1], s=7, alpha=0.38, color=CLASS_COLORS[class_id], label=class_name)
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=8, loc="upper left")

axes[2].scatter(target_reference[:, 0].cpu(), target_reference[:, 1].cpu(), s=8, alpha=0.10, color="#555555")
paired_indexes = torch.linspace(0, baseline_endpoints.shape[0] - 1, 70).long().to(device)
starts = baseline_endpoints[paired_indexes]
changes = aligned_endpoints[paired_indexes] - starts
axes[2].quiver(
    starts[:, 0].cpu(), starts[:, 1].cpu(), changes[:, 0].cpu(), changes[:, 1].cpu(),
    angles="xy", scale_units="xy", scale=1, width=0.004, color="#0072B2", alpha=0.55,
)
axes[2].set_title("Paired baseline-to-steered endpoint changes")
for ax in axes:
    ax.set(aspect="equal", xlim=(-4.5, 4.5), ylim=(-4.2, 4.5))
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

sweep = noise_alignment_table.dropna(subset=["strength"]).sort_values("strength")
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
metric_specs = [
    ("target_rate", "target-class rate (higher)", None),
    ("target_wasserstein", "Wasserstein distance to target (lower)", None),
    ("mean_error", "target mean error (lower)", None),
    ("diversity_ratio", "diversity ratio", 1.0),
]
for ax, (column, label, ideal) in zip(axes, metric_specs):
    ax.plot(sweep["strength"], sweep[column], marker="o", color="#0072B2")
    if ideal is not None:
        ax.axhline(ideal, color="#777777", ls="--", label="target value")
        ax.legend(frameon=False, fontsize=8)
    ax.set(xlabel="noise-alignment strength", ylabel=label)
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
baseline_row = noise_alignment_table.set_index("method").loc["baseline"]
aligned_row = noise_alignment_table.set_index("method").loc["noise alignment 1.6"]

control_gain = aligned_row["target_rate"] - baseline_row["target_rate"]
wasserstein_reduction = baseline_row["target_wasserstein"] - aligned_row["target_wasserstein"]
mean_error_reduction = baseline_row["mean_error"] - aligned_row["mean_error"]

alignment_evidence = pd.DataFrame({
    "question": [
        "zero strength reproduces baseline",
        "target-class rate improves by at least 0.15",
        "Wasserstein distance to target decreases by at least 0.20",
        "mean error decreases by at least 0.20",
        "resembles full class-conditional sampling",
    ],
    "observed": [
        zero_strength_error,
        control_gain,
        wasserstein_reduction,
        mean_error_reduction,
        aligned_row["target_rate"],
    ],
    "supported": [
        zero_strength_error < 1e-6,
        control_gain >= 0.15,
        wasserstein_reduction >= 0.20,
        mean_error_reduction >= 0.20,
        aligned_row["target_rate"] >= 0.90 and 0.5 <= aligned_row["diversity_ratio"] <= 2.0,
    ],
})
display(alignment_evidence.round(4))

partial_success = bool(alignment_evidence.loc[1:3, "supported"].all())
full_conditional = bool(alignment_evidence.loc[4, "supported"])
print("partial coarse steering supported:", partial_success)
print("full class-conditional generation supported:", full_conditional)
if partial_success:
    print("WHAT WORKED: target rate increased while target Wasserstein distance and mean error both decreased.")
if not full_conditional:
    print("WHAT DID NOT HAPPEN: off-target mass and excessive spread remain; this is not full class conditioning.")
print("WHAT THE METHOD IS NOT: no point was selected and no point-attraction controller was used.")


## Interpretation guardrails

- Higher target-class rate measures control, not sample quality.
- Lower Wasserstein distance measures agreement with target examples.
- A diversity ratio far below one indicates collapse; far above one indicates
  excessive spread.
- The model remains unconditional and unchanged. Only small class/full summary
  statistics are prepared offline.
- In high-dimensional image experiments, PCA gives a low-rank covariance model.
  In 2D we keep both principal directions, so the computation is a full Gaussian
  denoiser.
- The intended result is **partial coarse steering**: target rate, Wasserstein distance, and mean
  error should improve together. A high remaining diversity ratio or many
  off-target endpoints rules out a claim of full class-conditional generation.

**Questions**

1. Why should this coarse correction be most useful at high noise?
2. Which strength best balances target rate and target Wasserstein distance in your run?
3. What information is lost when a multimodal full dataset is approximated by one Gaussian?
